<a href="https://colab.research.google.com/github/attilagk/VPS13A-deep-learning/blob/main/notebooks/2025-11-13-DL-hello-world.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#%load_ext autoreload
#%autoreload 2
#%reload_ext autoreload
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor
from torchvision import datasets

In [2]:
torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0)

('2.12.0.dev20260304+cu128', True, 'NVIDIA GeForce RTX 5060 Ti')

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [4]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))
    print("arch list:", torch.cuda.get_arch_list())
    x = torch.randn(4096, 4096, device="cuda")
    y = torch.randn(4096, 4096, device="cuda")
    z = x @ y
    torch.cuda.synchronize()
    print("matmul ok:", z.shape)

torch: 2.12.0.dev20260304+cu128
cuda available: True
gpu: NVIDIA GeForce RTX 5060 Ti
capability: (12, 0)
arch list: ['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
matmul ok: torch.Size([4096, 4096])


# Hello World!

From the PyTorch tutorial: [Optimizing Model Parameters](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)

In [5]:
training_data = datasets.FashionMNIST(
    root='../data',
    train=True,
    download=True,
    transform=ToTensor(),
)

In [6]:
test_data = datasets.FashionMNIST(
    root='../data',
    train=False,
    download=True,
    transform=ToTensor(),
)

In [7]:
loader_kwargs = {"batch_size": 64}
if device == "cuda":
    loader_kwargs.update({"pin_memory": True, "num_workers": 4})

train_dataloader = DataLoader(training_data, shuffle=True, **loader_kwargs)
test_dataloader = DataLoader(test_data, batch_size=64)

In [8]:
class HelloWorldNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = HelloWorldNN().to(device)

In [9]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [10]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        # Compute loss
        pred = model(X)
        loss = loss_fn(pred, y)
        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        # Print progress
        if batch % 100 == 0:
            loss = loss.item()
            current = batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

        test_loss /= num_batches
        correct /= size
        print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [13]:
epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)

Epoch 1
-------------------------------
loss: 0.780085  [   64/60000]
loss: 0.750384  [ 6464/60000]
loss: 0.728407  [12864/60000]
loss: 0.798120  [19264/60000]
loss: 0.814490  [25664/60000]
loss: 0.760458  [32064/60000]
loss: 0.703699  [38464/60000]
loss: 0.838081  [44864/60000]
loss: 0.657090  [51264/60000]
loss: 0.835033  [57664/60000]
Test Error: 
 Accuracy: 71.4%, Avg loss: 0.766534 

Epoch 2
-------------------------------
loss: 0.672810  [   64/60000]
loss: 0.662041  [ 6464/60000]
loss: 0.771483  [12864/60000]
loss: 0.868127  [19264/60000]
loss: 0.706593  [25664/60000]
loss: 0.548645  [32064/60000]
loss: 0.877020  [38464/60000]
loss: 0.714601  [44864/60000]
loss: 0.918085  [51264/60000]
loss: 0.752299  [57664/60000]
Test Error: 
 Accuracy: 72.9%, Avg loss: 0.744024 

Epoch 3
-------------------------------
loss: 0.702933  [   64/60000]
loss: 0.652689  [ 6464/60000]
loss: 0.976875  [12864/60000]
loss: 0.632494  [19264/60000]
loss: 0.826803  [25664/60000]
loss: 0.652861  [32064/600

In [12]:
%connect_info

{
  "shell_port": 32979,
  "iopub_port": 59721,
  "stdin_port": 39825,
  "control_port": 42547,
  "hb_port": 35609,
  "ip": "127.0.0.1",
  "key": "4eae06aa-2921c1df538d5d4fab9820b8",
  "transport": "tcp",
  "signature_scheme": "hmac-sha256",
  "kernel_name": "dl-cuda-nightly",
  "jupyter_session": "/home/attila/projects/VPS13A-deep-learning/notebooks/2025-11-13-DL-hello-world.ipynb"
}

Paste the above JSON into a file, and connect with:
    $> jupyter <app> --existing <file>
or, if you are local, you can connect with just:
    $> jupyter <app> --existing kernel-c9927248-d05c-4288-81be-2e87c093f954.json
or even just:
    $> jupyter <app> --existing
if this is the most recent Jupyter kernel you have started.
